# Run 4: Baseline U-Net (Masked L1) + 3 Multi-Scale SN-PatchGAN (Branch C)

Fine-tune Generator từ `best_30ep.pt` với 3 bộ Discriminator đa tỷ lệ (`256²`, `128²`, `64²`).


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
!pip install pytorch-msssim


In [ ]:
import os, json, random, hashlib, time, platform, zipfile
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np
from PIL import Image
import torch
from torch import nn
import torch.nn.functional as F
from torch.nn.utils import spectral_norm
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms.functional import pil_to_tensor
from torchvision.utils import save_image
from pytorch_msssim import ssim
from tqdm.auto import tqdm

# ==============================================================================
# CẤU HÌNH NHÁNH C (Multi-Scale: 3 PatchGAN + D-Warmup + SpectralNorm, Ep 31->50)
# ==============================================================================
BASE_DIR = Path('/content/drive/MyDrive/ImageHarmonization/iHarmony4_dataset_resize_256')
if not BASE_DIR.exists():
    BASE_DIR = Path('/content/drive/MyDrive/ImageHarmonization')

PROJECT = Path('/content/drive/MyDrive/ImageHarmonization')
DATA = Path('/content/iharmony4_256')
ARCHIVES = ['HCOCO', 'HAdobe5k', 'HFlickr', 'Hday2night']
EXTS = {'.jpg', '.jpeg', '.png'}

BASE_CKPT = PROJECT / 'runs/unet_masked_l1_50ep_all4_bs32_seed42/best_30ep.pt'
RUN = PROJECT / 'runs/unet_branchC_3patchgan_50ep_all4_bs32_seed42'
RUN.mkdir(parents=True, exist_ok=True)

CFG = dict(
    branch='C',
    seed=42,
    start_epoch=30,
    total_epochs=50,
    extra_epochs=20,
    size=256,
    batch_size=32,
    workers=2,
    base_channels=32,
    num_D=3,
    ndf=64,
    lr_G=5e-5,
    lr_D=1e-4,
    eta_min=1e-6,
    lambda_l1=100.0,
    real_label=0.9,
    n_critic_warmup_epochs=3,
    val_fraction=0.05,
    loss='masked_l1_plus_3_multiscale_patchgan',
    architecture='unet_groupnorm_v1_sn_multiscale_patchgan',
)

DEVICE = torch.device('cuda')
random.seed(CFG['seed'])
np.random.seed(CFG['seed'])
torch.manual_seed(CFG['seed'])
torch.cuda.manual_seed_all(CFG['seed'])
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

# Tự giải nén dữ liệu nếu mở runtime mới
for subset in ARCHIVES:
    dest = DATA / subset
    if not (dest / '.extracted_ok').exists():
        zip_path = BASE_DIR / f'{subset}_256.zip'
        assert zip_path.is_file(), f'Không tìm thấy {zip_path}'
        print(f'Đang giải nén {zip_path.name}...')
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(DATA)
        (dest / '.extracted_ok').touch()

# 1. Lập chỉ mục & chia Train / Val / Test khớp 100% Nhánh A & B
def make_index(folder):
    return {p.stem: p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in EXTS and not p.name.startswith('.')}

records = {'train': [], 'val': [], 'test': []}
for subset in ARCHIVES:
    subset_root = DATA / subset
    comp = make_index(subset_root / 'composite_images')
    mask = make_index(subset_root / 'masks')
    real = make_index(subset_root / 'real_images')

    subsets = {}
    for split in ('train', 'test'):
        txt = subset_root / f'{subset}_{split}.txt'
        rows = []
        for line in txt.read_text(encoding='utf-8-sig').splitlines():
            if not line.strip():
                continue
            stem = Path(line.strip().replace('\\', '/').split()[0]).stem
            parts = stem.split('_')
            source_id, mask_id = '_'.join(parts[:-2]), '_'.join(parts[:-1])
            rows.append(dict(
                subset=subset, stem=stem, group=f'{subset}/{source_id}',
                comp=str(comp[stem]), mask=str(mask[mask_id]), real=str(real[source_id]),
            ))
        subsets[split] = rows

    groups = sorted({r['group'] for r in subsets['train']})
    random.Random(CFG['seed']).shuffle(groups)
    val_groups = set(groups[:max(1, round(len(groups) * CFG['val_fraction']))])

    for r in subsets['train']:
        records['val' if r['group'] in val_groups else 'train'].append(r)
    records['test'].extend(subsets['test'])

split_ids = {s: [f"{r['subset']}/{r['stem']}" for r in rows] for s, rows in records.items()}
fingerprint = hashlib.sha256(json.dumps(split_ids, sort_keys=True).encode()).hexdigest()
print(f"Train: {len(records['train'])} | Val: {len(records['val'])} | Test: {len(records['test'])}")


class HarmonyDataset(Dataset):
    def __init__(self, rows, train=False):
        self.rows, self.train = rows, train

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        r = self.rows[i]
        target_size = (CFG['size'], CFG['size'])
        images = []
        for key, mode in [('comp', 'RGB'), ('mask', 'L'), ('real', 'RGB')]:
            with Image.open(r[key]) as im:
                im = im.convert(mode)
                if im.size != target_size:
                    resample = Image.Resampling.NEAREST if key == 'mask' else Image.Resampling.BILINEAR
                    im = im.resize(target_size, resample)
                images.append(pil_to_tensor(im).float() / 255.)
        c, m, y = images
        m_bin = (m >= 0.5).float()
        m = m_bin if m_bin.sum() > 0 else (m > 0).float()
        if self.train and torch.rand(()) < 0.5:
            c, m, y = [x.flip(-1) for x in (c, m, y)]
        return c, m, y, r['subset']


def loader(split, epoch=0):
    g = torch.Generator().manual_seed(CFG['seed'] + epoch)
    is_train = (split == 'train')
    return DataLoader(
        HarmonyDataset(records[split], is_train),
        batch_size=CFG['batch_size'],
        shuffle=is_train,
        num_workers=CFG['workers'],
        pin_memory=True,
        drop_last=False,
        persistent_workers=(CFG['workers'] > 0),
        generator=g,
    )


# 2. Generator U-Net (khớp 100% checkpoint gốc)
class Block(nn.Sequential):
    def __init__(self, cin, cout):
        super().__init__(
            nn.Conv2d(cin, cout, 3, padding=1, bias=False),
            nn.GroupNorm(8, cout),
            nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1, bias=False),
            nn.GroupNorm(8, cout),
            nn.ReLU(inplace=True),
        )


class UNet(nn.Module):
    def __init__(self, b=32):
        super().__init__()
        self.enc = nn.ModuleList([Block(4, b), Block(b, b*2), Block(b*2, b*4), Block(b*4, b*8), Block(b*8, b*16)])
        self.up = nn.ModuleList([
            nn.ConvTranspose2d(b*16, b*8, 2, 2),
            nn.ConvTranspose2d(b*8, b*4, 2, 2),
            nn.ConvTranspose2d(b*4, b*2, 2, 2),
            nn.ConvTranspose2d(b*2, b, 2, 2),
        ])
        self.dec = nn.ModuleList([Block(b*16, b*8), Block(b*8, b*4), Block(b*4, b*2), Block(b*2, b)])
        self.head = nn.Conv2d(b, 3, 1)

    def forward(self, x):
        skips = []
        for i, block in enumerate(self.enc):
            x = block(x)
            if i < 4:
                skips.append(x)
                x = F.max_pool2d(x, 2)
        for up, block, skip in zip(self.up, self.dec, reversed(skips)):
            x = block(torch.cat([up(x), skip], 1))
        return self.head(x).sigmoid()


# 3. Kiến trúc Discriminator: Multi-Scale 3 PatchGAN (256x256, 128x128, 64x64)
class SinglePatchGAN(nn.Module):
    def __init__(self, input_nc=4, ndf=64):
        super().__init__()
        self.net = nn.Sequential(
            spectral_norm(nn.Conv2d(input_nc, ndf, 4, stride=2, padding=1)),
            nn.LeakyReLU(0.2, inplace=True),
            spectral_norm(nn.Conv2d(ndf, ndf * 2, 4, stride=2, padding=1, bias=False)),
            nn.InstanceNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),
            spectral_norm(nn.Conv2d(ndf * 2, ndf * 4, 4, stride=2, padding=1, bias=False)),
            nn.InstanceNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),
            spectral_norm(nn.Conv2d(ndf * 4, ndf * 8, 4, stride=1, padding=1, bias=False)),
            nn.InstanceNorm2d(ndf * 8),
            nn.LeakyReLU(0.2, inplace=True),
            spectral_norm(nn.Conv2d(ndf * 8, 1, 4, stride=1, padding=1)),
        )

    def forward(self, x):
        return self.net(x)


class MultiScalePatchGANDiscriminator(nn.Module):
    def __init__(self, input_nc=4, ndf=64, num_D=3):
        super().__init__()
        self.num_D = num_D
        self.discriminators = nn.ModuleList([
            SinglePatchGAN(input_nc=input_nc, ndf=ndf) for _ in range(num_D)
        ])
        self.downsample = nn.AvgPool2d(kernel_size=3, stride=2, padding=1, count_include_pad=False)

    def forward(self, x):
        outputs = []
        cur_x = x
        for i, disc in enumerate(self.discriminators):
            outputs.append(disc(cur_x))
            if i < self.num_D - 1:
                cur_x = self.downsample(cur_x)
        return outputs


def masked_l1_loss(pred, target, mask, eps=1e-6):
    diff = torch.abs(pred - target) * mask
    per_img = diff.sum(dim=(1, 2, 3)) / (3.0 * mask.sum(dim=(1, 2, 3)).clamp_min(eps))
    return per_img.mean()


@torch.inference_mode()
def evaluate(loader_, save_to=None, include_ssim=False):
    model.eval()
    sums = defaultdict(Counter)
    for c, m, y, subsets in tqdm(loader_, desc='evaluate', leave=False):
        c, m, y = [x.to(DEVICE, non_blocking=True) for x in (c, m, y)]
        with torch.autocast('cuda', dtype=torch.float16):
            raw = model(torch.cat([c, m], 1))
        raw = raw.float()
        out = raw * m + c * (1 - m)
        err = (out - y).square()
        mse = err.mean((1, 2, 3))
        fmse = (err * m).sum((1, 2, 3)) / (3 * m.sum((1, 2, 3)).clamp_min(1.0))

        values = dict(
            raw_L1=(raw - y).abs().mean((1, 2, 3)),
            MSE=mse * 255**2,
            fMSE=fmse * 255**2,
            PSNR=-10 * torch.log10(mse.clamp_min(1e-12)),
            fPSNR=-10 * torch.log10(fmse.clamp_min(1e-12)),
        )
        if include_ssim:
            values['SSIM'] = ssim(out, y, data_range=1, size_average=False)

        for i, subset in enumerate(subsets):
            for label in ('all', subset):
                sums[label]['n'] += 1
                for k, v in values.items():
                    sums[label][k] += v[i].item()

        if save_to is not None:
            panels = torch.stack([c, m.expand_as(c), raw, out, y], 1).flatten(0, 1)
            save_image(panels[:30], str(save_to), nrow=5)
            save_to = None

    return {s: {k: (v if k == 'n' else v / d['n']) for k, v in d.items()} for s, d in sums.items()}


def atomic_save(obj, target):
    tmp = target.with_name(target.name + '.tmp')
    torch.save(obj, tmp)
    os.replace(tmp, target)


# ==============================================================================
# KHỞI TẠO & TRAIN LOOP NHÁNH C (3 PATCHGAN)
# ==============================================================================
model = UNet(CFG['base_channels']).to(DEVICE)
netD = MultiScalePatchGANDiscriminator(input_nc=4, ndf=CFG['ndf'], num_D=CFG['num_D']).to(DEVICE)

opt_G = torch.optim.Adam(model.parameters(), lr=CFG['lr_G'], betas=(0.5, 0.999))
opt_D = torch.optim.Adam(netD.parameters(), lr=CFG['lr_D'], betas=(0.5, 0.999))

sched_G = torch.optim.lr_scheduler.CosineAnnealingLR(opt_G, T_max=CFG['extra_epochs'] - 1, eta_min=CFG['eta_min'])
sched_D = torch.optim.lr_scheduler.CosineAnnealingLR(opt_D, T_max=CFG['extra_epochs'], eta_min=CFG['eta_min'])

scaler_G = torch.amp.GradScaler('cuda')
scaler_D = torch.amp.GradScaler('cuda')

start_ep = 31
best_overall = float('inf')
best_gan = float('inf')
history = []

if (RUN / 'last.pt').exists():
    ck = torch.load(RUN / 'last.pt', map_location='cpu', weights_only=False)
    assert ck['cfg'] == CFG and ck['fingerprint'] == fingerprint, 'Config/data changed; use new RUN'
    model.load_state_dict(ck['model'])
    netD.load_state_dict(ck['netD'])
    opt_G.load_state_dict(ck['opt_G'])
    opt_D.load_state_dict(ck['opt_D'])
    for opt in (opt_G, opt_D):
        for state in opt.state.values():
            for k, v in state.items():
                if torch.is_tensor(v):
                    state[k] = v.to(DEVICE)
    sched_G.load_state_dict(ck['sched_G'])
    sched_D.load_state_dict(ck['sched_D'])
    scaler_G.load_state_dict(ck['scaler_G'])
    scaler_D.load_state_dict(ck['scaler_D'])
    start_ep = ck['epoch'] + 1
    best_overall = ck['best']
    best_gan = ck.get('best_gan', best_overall)
    history = ck['history']
    torch.set_rng_state(ck['torch_rng'])
    torch.cuda.set_rng_state_all(ck['cuda_rng'])
    print(f"Resume Branch C từ Epoch {start_ep}/50 (best_overall={best_overall:.2f}, best_gan={best_gan:.2f})")
else:
    assert BASE_CKPT.is_file(), f"Không tìm thấy checkpoint gốc: {BASE_CKPT}"
    base_ck = torch.load(BASE_CKPT, map_location='cpu', weights_only=False)
    assert base_ck['fingerprint'] == fingerprint, 'Split fingerprint không khớp!'
    model.load_state_dict(base_ck['model'])
    best_overall = base_ck['val']['all']['fMSE']
    print(f"Khởi tạo Branch C (3 PatchGAN) từ {BASE_CKPT} (Epoch {base_ck.get('epoch', 30)}, val fMSE={best_overall:.2f})")

(RUN / 'config.json').write_text(json.dumps(dict(
    cfg=CFG,
    fingerprint=fingerprint,
    source_ckpt=str(BASE_CKPT),
    gpu=torch.cuda.get_device_name(),
    torch=torch.__version__,
    python=platform.python_version(),
), indent=2))

for ep in range(start_ep, CFG['total_epochs'] + 1):
    torch.cuda.synchronize()
    tic = time.perf_counter()
    torch.cuda.reset_peak_memory_stats()

    is_d_warmup = (ep == 31)
    model.eval() if is_d_warmup else model.train()
    netD.train()

    tot_l1, tot_gan, tot_d, count = 0.0, 0.0, 0.0, 0
    tot_d_scales = [0.0] * CFG['num_D']
    n_critic = 2 if ep < 31 + CFG['n_critic_warmup_epochs'] else 1
    phase_str = "D-Warmup" if is_d_warmup else f"nD={n_critic}"
    bar = tqdm(loader('train', ep), desc=f'Branch C (3D) | Ep {ep}/50 ({phase_str})')

    for c, m, y, _ in bar:
        c, m, y = [x.to(DEVICE, non_blocking=True) for x in (c, m, y)]
        inp = torch.cat([c, m], dim=1)

        # 1. Cập nhật Multi-Scale Discriminator (D1: 256, D2: 128, D3: 64)
        for _ in range(n_critic):
            opt_D.zero_grad(set_to_none=True)
            with torch.autocast('cuda', dtype=torch.float16):
                with torch.no_grad():
                    raw_fake = model(inp)
                    comp_fake = raw_fake * m + c * (1.0 - m)
                preds_real = netD(torch.cat([y, m], dim=1))
                preds_fake = netD(torch.cat([comp_fake.detach(), m], dim=1))

                d_losses = []
                for pr, pf in zip(preds_real, preds_fake):
                    d_l = 0.5 * (
                        F.mse_loss(pr, torch.full_like(pr, CFG['real_label'])) +
                        F.mse_loss(pf, torch.zeros_like(pf))
                    )
                    d_losses.append(d_l)
                loss_D = sum(d_losses) / CFG['num_D']

            if not torch.isfinite(loss_D):
                raise RuntimeError('Non-finite loss_D; inspect data/AMP')

            scaler_D.scale(loss_D).backward()
            scaler_D.unscale_(opt_D)
            nn.utils.clip_grad_norm_(netD.parameters(), 1.0)
            scaler_D.step(opt_D)
            scaler_D.update()

        # 2. Cập nhật Generator G
        if is_d_warmup:
            with torch.no_grad(), torch.autocast('cuda', dtype=torch.float16):
                l1_val = masked_l1_loss(raw_fake, y, m)
            gan_val = 0.0
        else:
            opt_G.zero_grad(set_to_none=True)
            with torch.autocast('cuda', dtype=torch.float16):
                raw = model(inp)
                comp_out = raw * m + c * (1.0 - m)
                l1_val = masked_l1_loss(raw, y, m)

                preds_fake_g = netD(torch.cat([comp_out, m], dim=1))
                gan_loss = sum(F.mse_loss(pf, torch.ones_like(pf)) for pf in preds_fake_g) / CFG['num_D']

                loss_G = CFG['lambda_l1'] * l1_val + gan_loss
                gan_val = gan_loss.item()

            if not torch.isfinite(loss_G):
                raise RuntimeError('Non-finite loss_G; inspect data/AMP')

            scaler_G.scale(loss_G).backward()
            scaler_G.unscale_(opt_G)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler_G.step(opt_G)
            scaler_G.update()

        bs = c.shape[0]
        tot_l1 += l1_val.item() * bs
        tot_gan += gan_val * bs
        tot_d += loss_D.item() * bs
        for idx, dl in enumerate(d_losses):
            tot_d_scales[idx] += dl.item() * bs
        count += bs

        bar.set_postfix(
            L1=f'{tot_l1/count:.5f}',
            GAN=f'{tot_gan/count:.4f}',
            D=f'{tot_d/count:.4f}',
        )

    assert count == len(records['train']), f'Not all training samples were seen ({count} vs {len(records["train"])})'

    scores = evaluate(loader('val'), save_to=RUN / f'val_{ep:03d}.png')
    cur_fmse = scores['all']['fMSE']

    improved_overall = cur_fmse < best_overall
    best_overall = min(best_overall, cur_fmse)

    lr_g_used = opt_G.param_groups[0]['lr']
    lr_d_used = opt_D.param_groups[0]['lr']
    sched_D.step()
    if not is_d_warmup:
        sched_G.step()
    torch.cuda.synchronize()

    row = dict(
        branch='C',
        epoch=ep,
        train_L1=tot_l1 / count,
        train_GAN=tot_gan / count,
        train_D=tot_d / count,
        train_D_scales=[round(s / count, 5) for s in tot_d_scales],
        val=scores,
        lr_G=lr_g_used,
        lr_D=lr_d_used,
        seconds=time.perf_counter() - tic,
        peak_VRAM_GB=torch.cuda.max_memory_allocated() / 2**30,
    )
    history.append(row)

    payload = dict(model=model.state_dict(), netD=netD.state_dict(), cfg=CFG, epoch=ep, fingerprint=fingerprint, val=scores)

    if improved_overall:
        atomic_save(payload, RUN / 'best.pt')

    if not is_d_warmup and cur_fmse < best_gan:
        best_gan = cur_fmse
        atomic_save(payload, RUN / 'best_gan.pt')

    atomic_save(dict(
        epoch=ep,
        model=model.state_dict(),
        netD=netD.state_dict(),
        opt_G=opt_G.state_dict(),
        opt_D=opt_D.state_dict(),
        sched_G=sched_G.state_dict(),
        sched_D=sched_D.state_dict(),
        scaler_G=scaler_G.state_dict(),
        scaler_D=scaler_D.state_dict(),
        cfg=CFG,
        fingerprint=fingerprint,
        best=best_overall,
        best_gan=best_gan,
        history=history,
        torch_rng=torch.get_rng_state(),
        cuda_rng=torch.cuda.get_rng_state_all(),
    ), RUN / 'last.pt')
    (RUN / 'history.json').write_text(json.dumps(history, indent=2))
    print(json.dumps(row, ensure_ascii=False))

print(f'Done Branch C (3 PatchGAN)! Best overall fMSE: {best_overall:.2f} | Best GAN-phase (Ep 32-50) fMSE: {best_gan:.2f}')

Đang giải nén HCOCO_256.zip...
Đang giải nén HAdobe5k_256.zip...
Đang giải nén HFlickr_256.zip...
Đang giải nén Hday2night_256.zip...
Train: 62463 | Val: 3279 | Test: 7404
Khởi tạo Branch C (3 PatchGAN) từ /content/drive/MyDrive/ImageHarmonization/runs/unet_masked_l1_50ep_all4_bs32_seed42/best_30ep.pt (Epoch 29, val fMSE=397.88)


{"branch": "C", "epoch": 31, "train_L1": 0.05030149165650969, "train_GAN": 0.0, "train_D": 0.22208247306056308, "train_D_scales": [0.22404, 0.22256, 0.21965], "val": {"all": {"n": 3279, "raw_L1": 0.17709338885185813, "MSE": 40.5436619793398, "fMSE": 397.87979494314726, "PSNR": 36.00872848920249, "fPSNR": 24.34944521591917}, "HCOCO": {"n": 1924, "raw_L1": 0.17464156051141186, "MSE": 23.140737493357886, "fMSE": 369.6356207813889, "PSNR": 37.233579558295176, "fPSNR": 24.057863148235235}, "HAdobe5k": {"n": 974, "raw_L1": 0.1866478298999201, "MSE": 39.29364007374834, "fMSE": 324.07418924729194, "PSNR": 35.99174632281983, "fPSNR": 26.238237727594083}, "HFlickr": {"n": 368, "raw_L1": 0.165660142817575, "MSE": 128.65199368932971, "fMSE": 710.9841192908909, "PSNR": 29.943084141482476, "fPSNR": 21.110519061917845}, "Hday2night": {"n": 13, "raw_L1": 0.14776467245358688, "MSE": 215.68842950234045, "fMSE": 1244.4997699444111, "PSNR": 27.707520264845627, "fPSNR": 17.675969563997707}}, "lr_G": 5e-05,

{"branch": "C", "epoch": 32, "train_L1": 0.04955598227321294, "train_GAN": 0.3100881722522374, "train_D": 0.2032300215894088, "train_D_scales": [0.20269, 0.20396, 0.20304], "val": {"all": {"n": 3279, "raw_L1": 0.18139377525164319, "MSE": 41.14958146095422, "fMSE": 395.28630559289553, "PSNR": 36.01306057412583, "fPSNR": 24.360487373110068}, "HCOCO": {"n": 1924, "raw_L1": 0.1800524211660988, "MSE": 22.721553460290178, "fMSE": 362.1828875353331, "PSNR": 37.255101303053, "fPSNR": 24.07514526492097}, "HAdobe5k": {"n": 974, "raw_L1": 0.1890349633134978, "MSE": 41.05637405442506, "fMSE": 324.9134234202226, "PSNR": 35.97877419019382, "fPSNR": 26.25755868359513}, "HFlickr": {"n": 368, "raw_L1": 0.1693350210097497, "MSE": 132.10495923718682, "fMSE": 721.5080490319625, "PSNR": 29.898483131242834, "fPSNR": 21.06320230598035}, "Hday2night": {"n": 13, "raw_L1": 0.14876836366378343, "MSE": 200.74426342890814, "fMSE": 1332.560305081881, "PSNR": 27.849450918344353, "fPSNR": 17.795230792118954}}, "lr_G"

{"branch": "C", "epoch": 33, "train_L1": 0.04932809772587098, "train_GAN": 0.31802351708397897, "train_D": 0.19646321334228803, "train_D_scales": [0.19198, 0.19745, 0.19996], "val": {"all": {"n": 3279, "raw_L1": 0.17806292397514573, "MSE": 40.21529988795019, "fMSE": 391.1738836731115, "PSNR": 36.11072337391101, "fPSNR": 24.47793257690923}, "HCOCO": {"n": 1924, "raw_L1": 0.1760518777565624, "MSE": 22.4462983612583, "fMSE": 357.63901041451214, "PSNR": 37.3234605164637, "fPSNR": 24.16711570220281}, "HAdobe5k": {"n": 974, "raw_L1": 0.18672427322401894, "MSE": 38.215284199746485, "fMSE": 320.3964012858804, "PSNR": 36.1504627047623, "fPSNR": 26.4477182257102}, "HFlickr": {"n": 368, "raw_L1": 0.16678311586703942, "MSE": 131.7848117724061, "fMSE": 720.5985003657963, "PSNR": 29.96934385921644, "fPSNR": 21.138113674910173}, "Hday2night": {"n": 13, "raw_L1": 0.14606816550860038, "MSE": 227.75328790224515, "fMSE": 1331.95119535006, "PSNR": 27.4965149806096, "fPSNR": 17.43899418757512}}, "lr_G": 4.

{"branch": "C", "epoch": 34, "train_L1": 0.04911710859062381, "train_GAN": 0.33374708179577284, "train_D": 0.1958879647710612, "train_D_scales": [0.191, 0.19671, 0.19995], "val": {"all": {"n": 3279, "raw_L1": 0.18046816515918912, "MSE": 39.667705352548495, "fMSE": 391.6369555248797, "PSNR": 36.06898114738686, "fPSNR": 24.42304671497816}, "HCOCO": {"n": 1924, "raw_L1": 0.17870016428961452, "MSE": 22.709595602787953, "fMSE": 364.1123537670302, "PSNR": 37.28319952245066, "fPSNR": 24.11499320444595}, "HAdobe5k": {"n": 974, "raw_L1": 0.18926251519020088, "MSE": 37.92371342216429, "fMSE": 317.2483407266331, "PSNR": 36.082943125182354, "fPSNR": 26.36050917921125}, "HFlickr": {"n": 368, "raw_L1": 0.16767667159033212, "MSE": 126.93377774662298, "fMSE": 701.4442251246909, "PSNR": 29.97531615132871, "fPSNR": 21.140835202258565}, "Hday2night": {"n": 13, "raw_L1": 0.14533019409729883, "MSE": 209.83206363824698, "fMSE": 1268.7730618990386, "PSNR": 27.816181036142204, "fPSNR": 17.766150621267464}}, "

{"branch": "C", "epoch": 35, "train_L1": 0.04891506109225714, "train_GAN": 0.3373865327059592, "train_D": 0.19453488250446563, "train_D_scales": [0.18908, 0.1953, 0.19923], "val": {"all": {"n": 3279, "raw_L1": 0.1787600002548941, "MSE": 39.930550489366766, "fMSE": 390.6997996257841, "PSNR": 36.041830541593846, "fPSNR": 24.380642690248482}, "HCOCO": {"n": 1924, "raw_L1": 0.17618679973198073, "MSE": 22.63704291128567, "fMSE": 362.73258179091613, "PSNR": 37.27624356771457, "fPSNR": 24.101109090317312}, "HAdobe5k": {"n": 974, "raw_L1": 0.1886391177452198, "MSE": 39.582508705663486, "fMSE": 319.41480279605247, "PSNR": 35.988893082009696, "fPSNR": 26.227819163696477}, "HFlickr": {"n": 368, "raw_L1": 0.16750462869506166, "MSE": 126.94725094929986, "fMSE": 698.2620522975922, "PSNR": 29.990047288977582, "fPSNR": 21.157496032507524}, "Hday2night": {"n": 13, "raw_L1": 0.13803339291077393, "MSE": 162.20405112780057, "fMSE": 1164.3617260272686, "PSNR": 28.626958260169395, "fPSNR": 18.59531431931715

... (12 outputs left out)

{"branch": "C", "epoch": 46, "train_L1": 0.04691569919597909, "train_GAN": 0.3735958132755973, "train_D": 0.18028141130313305, "train_D_scales": [0.16838, 0.1834, 0.18907], "val": {"all": {"n": 3279, "raw_L1": 0.18143125872498933, "MSE": 39.996605748463494, "fMSE": 392.4825155606318, "PSNR": 36.05252935450846, "fPSNR": 24.411619684615488}, "HCOCO": {"n": 1924, "raw_L1": 0.1795176487539947, "MSE": 22.300589416973803, "fMSE": 358.49720985824996, "PSNR": 37.323655232570275, "fPSNR": 24.16677020641969}, "HAdobe5k": {"n": 974, "raw_L1": 0.18953760286498608, "MSE": 39.75610794882755, "fMSE": 328.7638460640055, "PSNR": 35.95040363844415, "fPSNR": 26.223105897648868}, "HFlickr": {"n": 368, "raw_L1": 0.17108656953939277, "MSE": 127.35366390645504, "fMSE": 703.3630406752876, "PSNR": 29.973008135090705, "fPSNR": 21.137248689713687}, "Hday2night": {"n": 13, "raw_L1": 0.1501275673508644, "MSE": 204.14913470928485, "fMSE": 1395.9962862454927, "PSNR": 27.6746883392334, "fPSNR": 17.617108198312614}}, 

{"branch": "C", "epoch": 47, "train_L1": 0.04683819693218549, "train_GAN": 0.3765876749719046, "train_D": 0.17907064192799446, "train_D_scales": [0.16674, 0.1824, 0.18807], "val": {"all": {"n": 3279, "raw_L1": 0.18054590358098638, "MSE": 39.99929641712072, "fMSE": 392.69916663601765, "PSNR": 36.061472437010195, "fPSNR": 24.420449502368495}, "HCOCO": {"n": 1924, "raw_L1": 0.17859069249068005, "MSE": 22.30336460025048, "fMSE": 358.1494425686382, "PSNR": 37.32552367089444, "fPSNR": 24.171217042046624}, "HAdobe5k": {"n": 974, "raw_L1": 0.18876592519652918, "MSE": 39.62676596757812, "fMSE": 328.9974966768856, "PSNR": 35.970996249627774, "fPSNR": 26.239884391212854}, "HFlickr": {"n": 368, "raw_L1": 0.17011100875780635, "MSE": 127.10498288792112, "fMSE": 705.6119008686231, "PSNR": 29.996173936387766, "fPSNR": 21.156028439169344}, "Hday2night": {"n": 13, "raw_L1": 0.14943562505336908, "MSE": 221.14736197544977, "fMSE": 1420.946047269381, "PSNR": 27.45524787902832, "fPSNR": 17.397420516380897}}

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7a5be30d2700>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1706, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1675, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7a5be30d2700>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1706, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1675, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/

{"branch": "C", "epoch": 48, "train_L1": 0.046764641332469464, "train_GAN": 0.3789751119241914, "train_D": 0.17803046591258453, "train_D_scales": [0.16536, 0.18144, 0.18729], "val": {"all": {"n": 3279, "raw_L1": 0.18137797853020715, "MSE": 39.629788699865195, "fMSE": 389.5634044908685, "PSNR": 36.07795886580411, "fPSNR": 24.43381000779394}, "HCOCO": {"n": 1924, "raw_L1": 0.17954471974929875, "MSE": 22.298656813753865, "fMSE": 357.4335028803026, "PSNR": 37.33523097702471, "fPSNR": 24.18035818534185}, "HAdobe5k": {"n": 974, "raw_L1": 0.1894981472751076, "MSE": 39.08011633999171, "fMSE": 323.8644584987687, "PSNR": 36.003455833732716, "fPSNR": 26.261630015206777}, "HFlickr": {"n": 368, "raw_L1": 0.1706248877124618, "MSE": 126.00783455371857, "fMSE": 697.6572215038797, "PSNR": 29.995144901068315, "fPSNR": 21.158202435659327}, "Hday2night": {"n": 13, "raw_L1": 0.1487074368275129, "MSE": 200.6573081750136, "fMSE": 1345.7310532789963, "PSNR": 27.77410888671875, "fPSNR": 17.723671986506535}}, "